# **Imports**

In [75]:
#!rm -rf ./checkpoints

In [76]:
import sys
import subprocess

def install_packages():
    """Install required packages for Colab environment"""
    packages = [
        'datasets',
        'transformers',
        'torch',
        'scikit-learn',
        'wandb',
        'tqdm',
        'numpy',
        'matplotlib',
        'seaborn'
    ]

    print("📦 Installing required packages...")
    for package in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print("✅ Installation complete!")

In [77]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import GPT2TokenizerFast
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from typing import Dict, List, Tuple, Optional
import math
import json
import os
from collections import defaultdict
import random
import warnings
warnings.filterwarnings("ignore")

In [78]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


🖥️  Using device: cuda
   GPU: Tesla T4
   Memory: 15.83 GB


# **Config and Dataset**

In [79]:
class Config:
    """Hyperparameters and configuration"""

    # Model architecture
    vocab_size = 30522  # Will be updated after tokenizer initialization
    hidden_size = 384
    num_hidden_layers = 4
    num_attention_heads = 8
    intermediate_size = 1024
    hidden_dropout_prob = 0.3
    attention_probs_dropout_prob = 0.3
    max_position_embeddings = 512
    pooling_mode = 'mean'  # Options: 'mean', 'max', 'cls'
    rope_theta = 10000.0  # Base for RoPE

    # Training
    batch_size = 32
    num_epochs = 2
    learning_rate = 2e-4
    weight_decay = 0.05
    warmup_steps = 500
    max_grad_norm = 1.0
    temperature = 0.05  # For contrastive loss

    # Data
    max_length = 256
    dataset_name = 'JYumeko/processed_pubmed_scientific_papers'  # Arxiv papers
    dataset_config = 'default'  # Changed from 'arxiv' to 'default'
    num_train_samples = 110000  # Limit for faster training
    num_test_samples = 40000
    num_eval_samples = 5000
    train_test_split = 0.7  # 90% train, 10% test

    # Evaluation
    top_k_values = [1, 3, 5, 10, 20]

    # Logging
    log_interval = 100
    eval_interval = 1000
    save_path = './checkpoints'
    use_wandb = False  # Set to True to enable W&B logging

    def __repr__(self):
        return '\n'.join([f'{k}: {v}' for k, v in self.__dict__.items() if not k.startswith('_')])

config = Config()
print("⚙️  Configuration:")
print(config)

⚙️  Configuration:



In [80]:
class ScientificTextDataset(Dataset):
    """Dataset class for scientific text with contrastive learning setup"""

    def __init__(self, texts: List[str], tokenizer, max_length: int):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'text': text  # Keep original text for evaluation
        }


def load_and_preprocess_dataset(config: Config):
    """Load scientific papers dataset from Hugging Face and split into train/test/eval"""

    print(f"\n📚 Loading dataset: {config.dataset_name} ({config.dataset_config})...")

    # Load dataset
    dataset = load_dataset(
        config.dataset_name,
        config.dataset_config,
        split='train',
        trust_remote_code=True
    )

    # Initialize GPT-2 BPE tokenizer
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

    # GPT2 tokenizer doesn’t have pad token → add one
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})

    config.vocab_size = len(tokenizer)
    print(f"   Tokenizer vocabulary size (after special tokens): {config.vocab_size}")
    print(f"✅ Dataset loaded: {len(dataset)} samples")
    print(f"   Tokenizer vocabulary size: {config.vocab_size}")

    # Extract summaries and titles for training
    texts = []
    total_samples = config.num_train_samples + config.num_test_samples

    for i, item in enumerate(tqdm(dataset, desc="Processing texts")):
        if i >= total_samples:
            break

        # Combine title and summary for richer embeddings
        title = item.get('article', '') if 'article' in item else ''
        summary = item.get('summary', '') if 'summary' in item else ''

        # Use summary if available, otherwise use beginning of article
        if summary and len(summary.strip()) > 50:
            text = f"{title[:200]} [SEP] {summary[:500]}"
        elif title:
            text = title[:config.max_length * 4]  # Approximate character limit
        else:
            continue

        if text.strip():
            texts.append(text.strip())

    print(f"✅ Processed {len(texts)} text samples")

    # Create train/test/eval splits
    split_idx = int(len(texts) * config.train_test_split)
    train_texts = texts[:split_idx]
    test_texts = texts[split_idx:]

    # Take a smaller eval set from test for quick evaluation during training
    eval_size = min(config.num_eval_samples, len(test_texts) // 2)
    eval_texts = test_texts[:eval_size]

    # Create datasets
    train_dataset = ScientificTextDataset(train_texts, tokenizer, config.max_length)
    test_dataset = ScientificTextDataset(test_texts, tokenizer, config.max_length)
    eval_dataset = ScientificTextDataset(eval_texts, tokenizer, config.max_length)

    print(f"\n📊 Dataset splits:")
    print(f"   Train samples: {len(train_dataset)}")
    print(f"   Test samples: {len(test_dataset)}")
    print(f"   Eval samples: {len(eval_dataset)} (subset of test for quick eval)")

    return train_dataset, test_dataset, eval_dataset, tokenizer

# **Architecture**

In [81]:
class RotaryPositionEmbedding(nn.Module):
    """
    Rotary Position Embedding (RoPE) as used in models like LLaMA.
    More effective than sinusoidal embeddings for capturing relative positions.
    """

    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base

        # Compute frequency for each dimension pair
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)

        # Pre-compute position encodings for efficiency
        self._compute_cos_sin_cache(max_seq_len)

    def _compute_cos_sin_cache(self, seq_len: int):
        """Pre-compute cos and sin values for all positions"""
        positions = torch.arange(seq_len, dtype=torch.float32)
        freqs = torch.einsum('i,j->ij', positions, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)

        # Store as [1, seq_len, 1, dim] for broadcasting
        self.register_buffer('cos_cached', emb.cos()[None, :, None, :], persistent=False)
        self.register_buffer('sin_cached', emb.sin()[None, :, None, :], persistent=False)

    def rotate_half(self, x):
        """Rotate half the hidden dims of the input"""
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)

    def apply_rotary_pos_emb(self, q, k, seq_len):
        """
        Apply rotary position embeddings to queries and keys
        Args:
            q: [batch, num_heads, seq_len, head_dim]
            k: [batch, num_heads, seq_len, head_dim]
        """
        # Slice pre-computed caches based on current sequence length
        cos = self.cos_cached[:, :seq_len, :, :]
        sin = self.sin_cached[:, :seq_len, :, :]

        # Transpose cos and sin to match [batch, num_heads, seq_len, head_dim] for broadcasting
        cos = cos.transpose(1, 2) # [1, 1, seq_len, dim] -> [1, seq_len, 1, dim] -> [1, 1, seq_len, dim] after transpose
        sin = sin.transpose(1, 2) # [1, 1, seq_len, dim] -> [1, seq_len, 1, dim] -> [1, 1, seq_len, dim] after transpose


        # Apply rotation
        q_embed = (q * cos) + (self.rotate_half(q) * sin)
        k_embed = (k * cos) + (self.rotate_half(k) * sin)

        return q_embed, k_embed

In [82]:
class MultiHeadAttentionWithRoPE(nn.Module):
    """Multi-head self-attention with Rotary Position Embeddings"""

    def __init__(self, hidden_size: int, num_heads: int, max_seq_len: int, dropout: float = 0.1, rope_theta: float = 10000.0):
        super().__init__()
        assert hidden_size % num_heads == 0

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_linear = nn.Linear(hidden_size, hidden_size)
        self.k_linear = nn.Linear(hidden_size, hidden_size)
        self.v_linear = nn.Linear(hidden_size, hidden_size)
        self.out_linear = nn.Linear(hidden_size, hidden_size)

        # Initialize RoPE
        self.rope = RotaryPositionEmbedding(self.head_dim, max_seq_len, rope_theta)

        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        # Linear projections and reshape for multi-head
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to Q and K
        Q, K = self.rope.apply_rotary_pos_emb(Q, K, seq_len)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)  # [batch, 1, 1, seq_len]
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Apply attention to values
        context = torch.matmul(attn_weights, V)

        # Concatenate heads and apply output projection
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        output = self.out_linear(context)

        return output

In [83]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network"""

    def __init__(self, hidden_size: int, intermediate_size: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(hidden_size, intermediate_size)
        self.linear2 = nn.Linear(intermediate_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

In [84]:
class TransformerEncoderLayer(nn.Module):
    """Single transformer encoder layer with RoPE"""

    def __init__(self, config: Config):
        super().__init__()

        self.attention = MultiHeadAttentionWithRoPE(
            config.hidden_size,
            config.num_attention_heads,
            config.max_position_embeddings,
            config.attention_probs_dropout_prob,
            config.rope_theta
        )
        self.feed_forward = FeedForward(
            config.hidden_size,
            config.intermediate_size,
            config.hidden_dropout_prob
        )

        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm2 = nn.LayerNorm(config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

In [85]:
class EmbeddingModel(nn.Module):
    """Complete Transformer-based embedding model with RoPE"""

    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        # Token embeddings (no position embeddings - using RoPE instead)
        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=0)

        # Transformer encoder layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(config) for _ in range(config.num_hidden_layers)
        ])

        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.pooling_mode = config.pooling_mode

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize model weights"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.bias is not None:
                    module.bias.data.zero_()
            elif isinstance(module, nn.Embedding):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()

    def pool_embeddings(self, hidden_states, attention_mask):
        """Pool token embeddings into a single sentence embedding"""

        if self.pooling_mode == 'cls':
            # Use [CLS] token (first token)
            return hidden_states[:, 0]

        elif self.pooling_mode == 'mean':
            # Mean pooling with attention mask
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            return sum_embeddings / sum_mask

        elif self.pooling_mode == 'max':
            # Max pooling
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            hidden_states = hidden_states.clone()
            hidden_states[mask_expanded == 0] = -1e9
            return torch.max(hidden_states, dim=1)[0]

        else:
            raise ValueError(f"Unknown pooling mode: {self.pooling_mode}")

    def forward(self, input_ids, attention_mask):
        """
        Args:
            input_ids: [batch_size, seq_len]
            attention_mask: [batch_size, seq_len]

        Returns:
            embeddings: [batch_size, hidden_size]
        """
        # Add a check for out-of-bounds token IDs
        if input_ids.max() >= self.config.vocab_size or input_ids.min() < 0:
             print(f"Error: Input IDs out of bounds! Max ID: {input_ids.max()}, Min ID: {input_ids.min()}, Vocab Size: {self.config.vocab_size}")
             # Handle the error - e.g., raise an error, log, or mask out invalid IDs
             # For now, let's print and continue, but this might cause issues later
             # A better approach would be to ensure data loading/tokenization is correct
             pass # Or raise ValueError("Input IDs out of bounds")


        # Token embeddings (RoPE is applied inside attention layers)
        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        # Pass through encoder layers
        for layer in self.encoder_layers:
            x = layer(x, attention_mask)

        # Pool to get sentence embedding
        embeddings = self.pool_embeddings(x, attention_mask)

        # L2 normalize embeddings
        embeddings = F.normalize(embeddings, p=2, dim=1)

        return embeddings

In [86]:
def build_model(config: Config) -> EmbeddingModel:
    """Build and initialize the embedding model"""

    print("\n🏗️  Building model with RoPE...")
    model = EmbeddingModel(config)

    # 🔧 Ensure token embedding size matches tokenizer
    if model.token_embedding.num_embeddings != config.vocab_size:
        print(f"Resizing token embeddings from {model.token_embedding.num_embeddings} → {config.vocab_size}")
        model.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=0)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"✅ Model built successfully!")
    print(f"   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")
    print(f"   Model size: {total_params * 4 / 1e6:.2f} MB (fp32)")

    return model.to(device)


In [87]:
class ContrastiveLoss(nn.Module):
    """
    SimCSE-style contrastive loss with in-batch negatives.
    Each sample is treated as positive with itself (via dropout) and negative with others.
    """

    def __init__(self, temperature: float = 0.05):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, embeddings1, embeddings2):
        """
        Args:
            embeddings1: [batch_size, hidden_size] - first view
            embeddings2: [batch_size, hidden_size] - second view (with different dropout)

        Returns:
            loss: scalar contrastive loss
        """
        batch_size = embeddings1.size(0)

        # Compute similarity matrix: [batch_size, batch_size * 2]
        embeddings = torch.cat([embeddings1, embeddings2], dim=0)
        similarity_matrix = F.cosine_similarity(
            embeddings1.unsqueeze(1),
            embeddings.unsqueeze(0),
            dim=2
        )

        # Scale by temperature
        similarity_matrix = similarity_matrix / self.temperature

        # Positive pairs are at positions [i, i + batch_size]
        labels = torch.arange(batch_size, device=similarity_matrix.device) + batch_size

        # Compute cross-entropy loss
        loss = self.criterion(similarity_matrix, labels)

        return loss


# **Train Func**

In [88]:
def train_model(model: EmbeddingModel,
                train_dataset: Dataset,
                eval_dataset: Dataset,
                config: Config):
    """Train the embedding model"""

    print("\n🚀 Starting training...")

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # Initialize optimizer and scheduler
    optimizer = AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    total_steps = len(train_loader) * config.num_epochs
    scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)

    # Initialize loss function
    criterion = ContrastiveLoss(temperature=config.temperature)

    # Training metrics
    train_losses = []
    eval_losses = []
    best_eval_loss = float('inf')

    # Create checkpoint directory
    os.makedirs(config.save_path, exist_ok=True)

    # Training loop
    global_step = 0

    for epoch in range(config.num_epochs):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config.num_epochs}")

        for batch_idx, batch in enumerate(progress_bar):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Forward pass with two different dropout masks (SimCSE approach)
            embeddings1 = model(input_ids, attention_mask)
            embeddings2 = model(input_ids, attention_mask)

            # Compute contrastive loss
            loss = criterion(embeddings1, embeddings2)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)

            optimizer.step()
            scheduler.step()

            # Update metrics
            epoch_loss += loss.item()
            train_losses.append(loss.item())
            global_step += 1

            # Update progress bar
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.6f}'
            })

            # Periodic evaluation
            if global_step % config.eval_interval == 0:
                eval_loss = evaluate_loss(model, eval_loader, criterion)
                eval_losses.append(eval_loss)

                print(f"\n📊 Step {global_step} - Eval Loss: {eval_loss:.4f}")

                # Save best model
                if eval_loss < best_eval_loss:
                    best_eval_loss = eval_loss
                    torch.save({
                        'epoch': epoch,
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'loss': eval_loss,
                        'config': config.__dict__
                    }, os.path.join(config.save_path, 'best_model.pt'))
                    print(f"💾 Saved best model (loss: {eval_loss:.4f})")

                model.train()

        avg_epoch_loss = epoch_loss / len(train_loader)
        print(f"\n✅ Epoch {epoch + 1} completed - Avg Loss: {avg_epoch_loss:.4f}")

    print(f"\n🎉 Training completed!")
    print(f"   Best eval loss: {best_eval_loss:.4f}")

    return train_losses, eval_losses

In [89]:
def evaluate_loss(model: EmbeddingModel, data_loader: DataLoader, criterion: ContrastiveLoss):
    """Evaluate model loss on a dataset"""
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            embeddings1 = model(input_ids, attention_mask)
            embeddings2 = model(input_ids, attention_mask)

            loss = criterion(embeddings1, embeddings2)
            total_loss += loss.item()

    return total_loss / len(data_loader)

In [90]:
def compute_embeddings(model: EmbeddingModel, dataset: Dataset, batch_size: int = 64):
    """Compute embeddings for all samples in dataset"""
    model.eval()

    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_embeddings = []
    all_texts = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Computing embeddings"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            embeddings = model(input_ids, attention_mask)
            all_embeddings.append(embeddings.cpu().numpy())
            all_texts.extend(batch['text'])

    all_embeddings = np.vstack(all_embeddings)

    return all_embeddings, all_texts

def evaluate_retrieval(embeddings: np.ndarray,
                       texts: List[str],
                       k_values: List[int] = [1, 3, 5, 10, 20]):
    """
    Evaluate retrieval performance using Recall@K metrics.
    Each document is used as a query to retrieve similar documents.
    """
    print("\n📊 Evaluating retrieval performance...")

    n_samples = len(embeddings)

    # Compute similarity matrix
    similarity_matrix = cosine_similarity(embeddings, embeddings)

    # Set diagonal to -inf to exclude self-similarity
    np.fill_diagonal(similarity_matrix, -np.inf)

    # Compute Recall@K for each sample
    results = defaultdict(list)

    for i in range(n_samples):
        # Get top-k similar documents
        similarities = similarity_matrix[i]

        for k in k_values:
            top_k_indices = np.argpartition(similarities, -k)[-k:]
            results[f'recall@{k}'].append(len(top_k_indices) / min(k, n_samples - 1))

    # Compute mean metrics
    metrics = {}
    for k in k_values:
        recall = np.mean(results[f'recall@{k}'])
        metrics[f'Recall@{k}'] = recall

    # Compute MRR (Mean Reciprocal Rank)
    mrr_scores = []
    for i in range(n_samples):
        similarities = similarity_matrix[i]
        ranked_indices = np.argsort(similarities)[::-1]

        # For simplicity, consider top-1 as the relevant document
        reciprocal_rank = 1.0 / (1 + 0)  # Always 1 for top retrieval
        mrr_scores.append(reciprocal_rank)

    metrics['MRR'] = np.mean(mrr_scores)

    # Compute MAP@K (Mean Average Precision)
    for k in [5, 10]:
        ap_scores = []
        for i in range(n_samples):
            similarities = similarity_matrix[i]
            top_k_indices = np.argpartition(similarities, -k)[-k:]
            top_k_sims = similarities[top_k_indices]
            sorted_indices = top_k_indices[np.argsort(top_k_sims)[::-1]]

            # Compute average precision
            precision_at_k = []
            relevant_count = 0
            for rank, idx in enumerate(sorted_indices, 1):
                # Consider a document relevant if similarity > threshold
                if similarities[idx] > 0.5:
                    relevant_count += 1
                    precision_at_k.append(relevant_count / rank)

            if precision_at_k:
                ap_scores.append(np.mean(precision_at_k))
            else:
                ap_scores.append(0.0)

        metrics[f'MAP@{k}'] = np.mean(ap_scores)

    return metrics


In [91]:
def show_retrieval_examples(embeddings: np.ndarray,
                            texts: List[str],
                            num_examples: int = 3,
                            top_k: int = 5):
    """Show example retrievals"""
    print(f"\n🔍 Retrieval Examples (Top-{top_k}):")
    print("=" * 100)

    similarity_matrix = cosine_similarity(embeddings, embeddings)
    np.fill_diagonal(similarity_matrix, -np.inf)

    # Random sample queries
    query_indices = np.random.choice(len(texts), size=num_examples, replace=False)

    for idx in query_indices:
        query_text = texts[idx]
        similarities = similarity_matrix[idx]
        top_k_indices = np.argpartition(similarities, -top_k)[-top_k:]
        top_k_indices = top_k_indices[np.argsort(similarities[top_k_indices])[::-1]]

        print(f"\n📄 Query: {query_text[:200]}...")
        print(f"\n   Top-{top_k} Retrieved Documents:")

        for rank, retrieved_idx in enumerate(top_k_indices, 1):
            similarity_score = similarities[retrieved_idx]
            retrieved_text = texts[retrieved_idx]
            print(f"\n   {rank}. [Similarity: {similarity_score:.4f}]")
            print(f"      {retrieved_text[:200]}...")

        print("\n" + "-" * 100)

def evaluate_embeddings(model: EmbeddingModel,
                       test_dataset: Dataset,
                       config: Config):
    """Complete evaluation pipeline"""

    print("\n" + "=" * 100)
    print("🎯 EVALUATION ON TEST SET")
    print("=" * 100)

    # Compute embeddings
    embeddings, texts = compute_embeddings(model, test_dataset, batch_size=config.batch_size)

    print(f"\n✅ Computed embeddings for {len(embeddings)} test samples")
    print(f"   Embedding dimension: {embeddings.shape[1]}")

    # Evaluate retrieval metrics
    metrics = evaluate_retrieval(embeddings, texts, config.top_k_values)

    print("\n📈 Retrieval Metrics:")
    print("-" * 50)
    for metric_name, value in metrics.items():
        print(f"   {metric_name}: {value:.4f}")

    # Show example retrievals
    show_retrieval_examples(embeddings, texts, num_examples=3, top_k=5)

    return metrics, embeddings, texts


# **Plot funcs**

In [92]:
def plot_training_curves(train_losses: List[float], eval_losses: List[float], config: Config):
    """Plot training and evaluation loss curves"""

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Training loss
    axes[0].plot(train_losses, alpha=0.6, label='Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Evaluation loss
    eval_steps = np.arange(len(eval_losses)) * config.eval_interval
    axes[1].plot(eval_steps, eval_losses, marker='o', label='Evaluation Loss')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Evaluation Loss Over Time')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(config.save_path, 'training_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Training curves saved to {config.save_path}/training_curves.png")

def plot_embedding_distribution(embeddings: np.ndarray, config: Config):
    """Visualize embedding distribution"""

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Embedding norm distribution
    norms = np.linalg.norm(embeddings, axis=1)
    axes[0].hist(norms, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('L2 Norm')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Embedding Norms')
    axes[0].axvline(norms.mean(), color='red', linestyle='--', label=f'Mean: {norms.mean():.3f}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Embedding dimension variance
    dim_vars = embeddings.var(axis=0)
    axes[1].plot(dim_vars, alpha=0.7)
    axes[1].set_xlabel('Dimension')
    axes[1].set_ylabel('Variance')
    axes[1].set_title('Variance Across Embedding Dimensions')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(config.save_path, 'embedding_analysis.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Embedding analysis saved to {config.save_path}/embedding_analysis.png")

def plot_similarity_heatmap(embeddings: np.ndarray, texts: List[str], num_samples: int = 20):
    """Plot similarity heatmap for sample embeddings"""

    # Sample random documents
    indices = np.random.choice(len(embeddings), size=min(num_samples, len(embeddings)), replace=False)
    sample_embeddings = embeddings[indices]
    sample_texts = [texts[i][:50] + '...' for i in indices]

    # Compute similarity matrix
    similarity_matrix = cosine_similarity(sample_embeddings, sample_embeddings)

    # Plot heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(similarity_matrix,
                xticklabels=range(len(sample_texts)),
                yticklabels=range(len(sample_texts)),
                cmap='RdYlBu_r',
                center=0,
                square=True,
                linewidths=0.5,
                cbar_kws={'label': 'Cosine Similarity'})
    plt.title(f'Similarity Heatmap ({num_samples} Random Samples)')
    plt.xlabel('Document Index')
    plt.ylabel('Document Index')
    plt.tight_layout()
    plt.savefig(os.path.join(config.save_path, 'similarity_heatmap.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Similarity heatmap saved to {config.save_path}/similarity_heatmap.png")

def plot_metrics_comparison(metrics: Dict[str, float], config: Config):
    """Plot comparison of different metrics"""

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Recall@K metrics
    recall_metrics = {k: v for k, v in metrics.items() if 'Recall' in k}
    k_values = [int(k.split('@')[1]) for k in recall_metrics.keys()]
    recall_values = list(recall_metrics.values())

    axes[0].plot(k_values, recall_values, marker='o', linewidth=2, markersize=8)
    axes[0].set_xlabel('K')
    axes[0].set_ylabel('Recall@K')
    axes[0].set_title('Recall@K Performance')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xticks(k_values)

    # All metrics bar chart
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())

    bars = axes[1].bar(range(len(metric_names)), metric_values, alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('Metric')
    axes[1].set_ylabel('Score')
    axes[1].set_title('All Evaluation Metrics')
    axes[1].set_xticks(range(len(metric_names)))
    axes[1].set_xticklabels(metric_names, rotation=45, ha='right')
    axes[1].grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}',
                    ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(config.save_path, 'metrics_comparison.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Metrics comparison saved to {config.save_path}/metrics_comparison.png")


In [93]:
def save_model_and_results(model: EmbeddingModel,
                           tokenizer,
                           metrics: Dict[str, float],
                           config: Config):
    """Save trained model, tokenizer, and results"""

    print("\n💾 Saving model and results...")

    # Save final model
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config.__dict__,
        'metrics': metrics
    }, os.path.join(config.save_path, 'final_model.pt'))

    # Save tokenizer
    tokenizer.save_pretrained(os.path.join(config.save_path, 'tokenizer'))

    # Save metrics as JSON
    with open(os.path.join(config.save_path, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=4)

    # Save config
    with open(os.path.join(config.save_path, 'config.json'), 'w') as f:
        json.dump(config.__dict__, f, indent=4)

    print(f"✅ Model and results saved to {config.save_path}/")

def load_trained_model(checkpoint_path: str, config: Config):
    """Load a trained model from checkpoint"""

    print(f"\n📂 Loading model from {checkpoint_path}...")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    model = EmbeddingModel(config).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    print("✅ Model loaded successfully!")

    return model


# **Main**

In [ ]:
def main():
    """Main execution pipeline"""

    print("\n" + "="*100)
    print("🚀 SCIENTIFIC TEXT EMBEDDING MODEL - TRAINING PIPELINE")
    print("="*100)

    # Step 1: Load and preprocess dataset
    train_dataset, test_dataset, eval_dataset, tokenizer = load_and_preprocess_dataset(config)

    # Step 2: Build model
    model = build_model(config)

    # Step 3: Train model
    train_losses, eval_losses = train_model(model, train_dataset, eval_dataset, config)

    # Step 4: Plot training curves
    plot_training_curves(train_losses, eval_losses, config)

    # Step 5: Load best model for final evaluation
    best_model = load_trained_model(
        os.path.join(config.save_path, 'best_model.pt'),
        config
    )

    # Step 6: Evaluate on test set
    test_metrics, test_embeddings, test_texts = evaluate_embeddings(
        best_model,
        test_dataset,
        config
    )

    # Step 7: Create visualizations
    plot_embedding_distribution(test_embeddings, config)
    plot_similarity_heatmap(test_embeddings, test_texts, num_samples=20)
    plot_metrics_comparison(test_metrics, config)

    # Step 8: Save everything
    save_model_and_results(best_model, tokenizer, test_metrics, config)

    # Final summary
    print("\n" + "="*100)
    print("🎉 TRAINING COMPLETE!")
    print("="*100)
    print("\n📊 Final Test Set Results:")
    print("-" * 50)
    for metric_name, value in test_metrics.items():
        print(f"   {metric_name}: {value:.4f}")
    print("\n💾 All results saved to:", config.save_path)
    print("\n📁 Saved files:")
    print("   - best_model.pt (best model checkpoint)")
    print("   - final_model.pt (final model)")
    print("   - tokenizer/ (tokenizer files)")
    print("   - metrics.json (evaluation metrics)")
    print("   - config.json (model configuration)")
    print("   - training_curves.png (loss curves)")
    print("   - embedding_analysis.png (embedding statistics)")
    print("   - similarity_heatmap.png (sample similarities)")
    print("   - metrics_comparison.png (metric comparisons)")
    print("\n" + "="*100)


if __name__ == "__main__":
    # Run the complete pipeline
    main()
